In [ ]:
# Imports
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
hltau_c= SkyCoord("4h31m38.43s", "+18d13m57.19s", frame='fk5')
hltau_ref = hltau_c.skyoffset_frame()
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
iras2a_ref = iras2a_c.skyoffset_frame()
distance_hltau = 147 #parsecs
distance_iras2a = 293 #parsecs
# choose which distance
distance = distance_iras2a

# cubefile = 'test_data/HLTau/HLTAU_HCOp32.fits'
# file_Tpeak = 'test_data/HLTau/HLTAU_HCOp32_Tpeak.fits'
cubefile = 'test_data/IRAS2A/D2CO_streamer_cluster_data.fits'
file_Tpeak = 'test_data/IRAS2A/D2CO_streamer_cluster_tpeak.fits'

# some constants
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km



### Original cube with spectra: Prepare the 1D streamer emission from the cube

In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)

# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer

'''
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer
'''

vmin = 6
vmax = 8
xmin = -5
xmax = 5
ymin = -12
ymax = 0.5
rms_thresh = 4

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 


In [ ]:
n_points = 10 # the number of points we want to reduce the data to


# Extract 1D streamline from the data cube
pc_coords, pc_means, pc_stds = extract_streamline.reduce_to_1D(streamer_cube, n_elements=n_points)
print(f"point cloud velocities (km/s): {pc_coords[2]}")

# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)
print(f"data velocities (km/s): {v_data}")

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]


data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

In [ ]:
# Helper function to compute and plot radial bin edges
def get_radial_bin_partitions(pc_coords, n_elements):
    """
    Compute the radial bin partition boundaries used in reduce_to_1D.
    
    Parameters
    ----------
    pc_coords : array of shape (3, n_points)
        Point cloud coordinates. Index 0 = RA, Index 1 = Dec, Index 2 = velocity
    n_elements : int
        Number of elements used in the reduction
    
    Returns
    -------
    partitions : array of shape (n_elements + 1,)
        Radial bin boundaries in arcsec
    """
    import numpy as np
    ra_coords = pc_coords[0]
    dec_coords = pc_coords[1]
    # Compute radial distance metric (same as in extract_streamline.get_distance_metric)
    distance_metric = np.sqrt(ra_coords**2 + dec_coords**2)
    # Compute percentile boundaries
    b_per = np.linspace(0, 100, n_elements + 1)
    partitions = np.array([np.percentile(distance_metric, per) for per in b_per])
    return partitions

def plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5):
    """
    Plot circles showing the radial bin edges on an RA-Dec plot.
    Axis limits are preserved after adding the circles.
    
    Parameters
    ----------
    ax : matplotlib axes object
        The axes to plot on
    partitions : array
        Radial bin boundaries in arcsec
    color : str
        Color of the circles
    linewidth : float
        Line width of the circles
    alpha : float
        Transparency of the circles
    """
    import matplotlib.patches as patches
    # Save current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Add circles
    for partition in partitions:
        circle = patches.Circle((0, 0), partition, fill=False, edgecolor=color, 
                               linewidth=linewidth, alpha=alpha)
        ax.add_patch(circle)
    
    # Restore original axis limits
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
# plot it
import matplotlib.pyplot as plt
# plot the observed data points as a scatter
plt.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
# plot the extracted 1D streamline with error bars
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
# plot the star
plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
# plot the radial bin edges
partitions = get_radial_bin_partitions(pc_coords, n_points)
ax = plt.gca()
plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
# flip the x axis to match the astronomical convention (RA increases to the left)
plt.gca().invert_xaxis()
plt.legend()
plt.title('Extracted 1D Streamer emission')


### Initial guess at parameters, and plot

The plot is so you can refine your initial parameters a bit. The comparison between stream_lines and stream_lines_grad is just a sanity check and will be removed

In [ ]:

# Initial guesses at params

# IRAS 2A D2CO

r0 = 3000*u.au
theta0 = 40.*u.deg
phi0 = 80.*u.deg
omega0 = 4e-13/u.s
v_r0 = 0.0*u.km/u.s
#params we set
Mstar = 4.0*u.Msun
inc = -45*u.deg
PA_ang = 194*u.deg #228
v_lsr = 7.5*u.km/u.s
'''


# approx HL Tau
r0 = 500 * u.au
theta0 = 75. * u.deg
phi0 = 260. * u.deg
omega0 = 2e-12 / u.s
v_r0 = 0.0 * u.km / u.s
Mstar = 2.1*u.Msun
inc = -47*u.deg
PA_ang = (48)*u.deg # note disk major axis PA is 138 (reported in paper), but we need minor axis
v_lsr = 7.1*u.km/u.s


# per-50 parameters
r0 = 3330.0 * u.au
theta0 = 61.5 * u.deg
phi0 = 28.0 * u.deg
v_r0 = 1.25 * u.km/u.s
omega0 = 4.53e-13 / u.s
inc = -67.0 * u.deg
PA_ang = (170.0) * u.deg
v_lsr = 8.0*u.km/u.s #idk about this one but it doesn't matter 
'''
# Run the forward model with initial params - using stream_lines
# x = RA offset (au), y = velocity (km/s), z = Dec offset (au)
print("Running stream_lines...")
(x1, y1, z1), (vx1, vy1, vz1) = stream_lines.xyz_stream(
                mass=Mstar, r0=r0, theta0=theta0, phi0=phi0,
                omega=omega0, v_r0=v_r0, inc=inc, pa=PA_ang,
                rmin=0.5e2*u.au, deltar=40*u.au) #<- decrease deltar for more accurate streamer calculation
dra_stream1 = -x1.value / distance * u.arcsec 
ddec_stream1 = z1.value / distance * u.arcsec
fil1 = SkyCoord(dra_stream1, ddec_stream1, frame=hltau_ref).transform_to(FK5)

print("---------------------------------------------")

# Run the forward model with initial params - using stream_lines_grad
# convert angles to radians, and strip units from all quantities
r0 = r0.to(u.au).value
theta0 = theta0.to(u.rad).value
phi0 = phi0.to(u.rad).value
omega0 = omega0.to(1/u.s).value
v_r0 = v_r0.to(u.km/u.s).value
inc = inc.to(u.rad).value
PA_ang = PA_ang.to(u.rad).value
Mstar = Mstar.to(u.Msun).value
v_lsr = v_lsr.to(u.km/u.s).value
# x = RA offset (au), y = velocity (km/s), z = Dec offset (au)
print("Running stream_lines_grad...")
(x2, y2, z2), (vx2, vy2, vz2) = stream_lines_grad.xyz_stream(
                mass=Mstar, r0=r0, theta0=theta0, phi0=phi0,
                omega=omega0, v_r0=v_r0, inc=inc, pa=PA_ang,
                rmin=0.5e2, deltar=40) #<- decrease deltar for more accurate streamer calculation
dra_stream2 = -np.array(x2, dtype=np.float64) / distance * u.arcsec # note the _value to get raw number for stream_lines_grad
ddec_stream2 = np.array(z2, dtype=np.float64) / distance * u.arcsec
fil2 = SkyCoord(dra_stream2, ddec_stream2, frame=hltau_ref).transform_to(FK5)

print("Plotting results...")
# Load the moment 0 data
hdu = fits.open(file_Tpeak)[0]
tpeak_data = hdu.data.squeeze()
wcs_Tpeak = WCS(hdu.header)

# Create figure with two subplots side by side
plt.close('all')
fig = plt.figure(figsize=(14, 6))

# Left subplot - stream_lines
ax1 = WCSAxes(fig, [0.05, 0.1, 0.4, 0.8], wcs=wcs_Tpeak.celestial)
fig.add_axes(ax1)
im1 = ax1.imshow(tpeak_data, cmap='inferno', vmin=0)
ax1.scatter(iras2a_c.ra, iras2a_c.dec, marker='*', transform=ax1.get_transform('world'),
    facecolor='white', edgecolor='black')
ax1.plot(fil1.ra, fil1.dec, color='black', transform=ax1.get_transform('world'), linewidth=5)
ax1.plot(fil1.ra, fil1.dec, color='red', transform=ax1.get_transform('world'), linewidth=2)
ax1.set_title('stream_lines')

# Right subplot - stream_lines_grad
ax2 = WCSAxes(fig, [0.52, 0.1, 0.4, 0.8], wcs=wcs_Tpeak.celestial)
fig.add_axes(ax2)
im2 = ax2.imshow(tpeak_data, cmap='inferno', vmin=0)
# ax2.scatter(hltau_c.ra, hltau_c.dec, marker='*', transform=ax2.get_transform('world'),
#     facecolor='white', edgecolor='black')
ax2.scatter(iras2a_c.ra, iras2a_c.dec, marker='*', transform=ax2.get_transform('world'),
    facecolor='white', edgecolor='black')
ax2.plot(fil2.ra, fil2.dec, color='black', transform=ax2.get_transform('world'), linewidth=5)
ax2.plot(fil2.ra, fil2.dec, color='red', transform=ax2.get_transform('world'), linewidth=2)
ax2.set_title('stream_lines_grad')


plt.show()



### TEST Forward model and calculating loss (with extra plots)

First set your input params in the format required

In [ ]:
# Parameters to optimize
initial_opt_params = {
    'r0': 3000.0,  # au
    'theta0': 40.0,  # degrees
    'phi0': 80.0,  # degrees
    'log_omega': np.log(4e-13),  # log(1/s)
    'v_r0': 0.0,  # km/s
}

# Fixed parameters (not optimized)
# HL Tau
# pa = 138 + 270 = 408 degrees, which is equivalent to 48 degrees (since PA is modulo 360)

'''
fixed_params = {
    'mass': 2.1,  # solar masses
    'inc': -47.0,  # degrees
    'pa': 48.0,  # degrees
    'rmin': 50.0,  # au
    'deltar': 40.0,  # au
    'v_lsr': 7.1  # km/s (systemic velocity)
}
'''

# IRAS2A
fixed_params = {
    'mass': 4.0,  # solar masses
    'inc': -45.0,  # degrees
    'pa': 194.0,  # degrees
    'rmin': 50.0,  # au
    'deltar':50.0,  # au
    'v_lsr': 7.5  # km/s (systemic velocity)
}


# Convert angles from degrees to radians
initial_opt_params['theta0'] = np.radians(initial_opt_params['theta0'])
initial_opt_params['phi0'] = np.radians(initial_opt_params['phi0'])
fixed_params['inc'] = np.radians(fixed_params['inc'])
fixed_params['pa'] = np.radians(fixed_params['pa'])

Doing chi2 loss more manually, just so we can check the model point choosing is working properly (delete this later)

In [ ]:
# forward model (same inputs as chi2_loss)
ra_model, dec_model, v_model = gradient_descent.forward_model(
    initial_opt_params, fixed_params, distance
)

# keep finite points for plotting only
finite_model = jnp.isfinite(ra_model) & jnp.isfinite(dec_model) & jnp.isfinite(v_model)
ra_model_plot = ra_model[finite_model]
dec_model_plot = dec_model[finite_model]
v_model_plot = v_model[finite_model]

# match model to data exactly as chi2_loss does
ra_model_interp, dec_model_interp, v_model_interp, valid = gradient_descent.match_model_to_data_curve(
    ra_model, dec_model, v_model, ra_data, dec_data
)

print(initial_opt_params)
print(fixed_params)
print(f"ra_data = {ra_data}")
print(f"ra_model_interp = {ra_model_interp}")
print(f"retained {int(jnp.sum(valid))}/{len(valid)} data points after overlap filtering")

# ---- Manual chi2_loss calculation (same logic as gradient_descent.chi2_loss) ----
# ---- Manual chi2_loss calculation (matches gradient_descent.chi2_loss) ----
# Coerce to float64 and floor sigmas to avoid division by zero
ra_data_f = jnp.asarray(ra_data, dtype=jnp.float64)
dec_data_f = jnp.asarray(dec_data, dtype=jnp.float64)
v_data_f = jnp.asarray(v_data, dtype=jnp.float64)

ra_sigma_f = jnp.asarray(ra_sigma, dtype=jnp.float64)
dec_sigma_f = jnp.asarray(dec_sigma, dtype=jnp.float64)
v_sigma_f = jnp.asarray(v_sigma, dtype=jnp.float64)

eps = jnp.asarray(1e-8, dtype=jnp.float64)
ra_sigma_safe = jnp.maximum(ra_sigma_f, eps)
dec_sigma_safe = jnp.maximum(dec_sigma_f, eps)
v_sigma_safe = jnp.maximum(v_sigma_f, eps)

# Recompute distance metrics and overlap domain exactly like chi2_loss
dmetric_data = extract_streamline.get_distance_metric(ra_data_f, dec_data_f)
dmetric_model = extract_streamline.get_distance_metric(ra_model, dec_model)

model_finite = jnp.isfinite(dmetric_model)
data_finite = jnp.isfinite(dmetric_data)

model_min = jnp.min(jnp.where(model_finite, dmetric_model, jnp.inf))
model_max = jnp.max(jnp.where(model_finite, dmetric_model, -jnp.inf))
data_min = jnp.min(jnp.where(data_finite, dmetric_data, jnp.inf))
data_max = jnp.max(jnp.where(data_finite, dmetric_data, -jnp.inf))

overlap_min = jnp.maximum(model_min, data_min)
overlap_max = jnp.minimum(model_max, data_max)

margin = jnp.asarray(0.05, dtype=jnp.float64)

dist_to_overlap = jnp.minimum(
    jnp.abs(dmetric_data - overlap_min),
    jnp.abs(dmetric_data - overlap_max)
)
weights = jnp.exp(- (dist_to_overlap / margin) ** 2)

penalty = (
    jnp.maximum(0.0, overlap_min - dmetric_data)
    + jnp.maximum(0.0, dmetric_data - overlap_max)
)
chi2_penalty = jnp.sum((penalty / margin) ** 2)

# Polar-space sky residual and velocity residual
r_data, theta_data = extract_streamline.cartesian_to_polar(ra_data_f, dec_data_f)
_, theta_model = extract_streamline.cartesian_to_polar(ra_model_interp, dec_model_interp)

dtheta = extract_streamline._wrap_to_pi(theta_data - theta_model)
dsky = r_data * dtheta
sigma_dsky = jnp.sqrt(ra_sigma_safe**2 + dec_sigma_safe**2)

chi2_dsky = jnp.sum(weights * ((dsky / sigma_dsky) ** 2))
chi2_v = jnp.sum(weights * (((v_data_f - v_model_interp) / v_sigma_safe) ** 2))
chi2_total = chi2_dsky + chi2_v + chi2_penalty

print(
    f"Chi2 dsky: {chi2_dsky:.2f}, Chi2 v: {chi2_v:.2f}, "
    f"Chi2 penalty: {chi2_penalty:.2f}, Total: {chi2_total:.2f}"
)
print(f"Overlap range in distance metric: [{float(overlap_min):.4f}, {float(overlap_max):.4f}]")
# ---- Plot ----
import matplotlib.pyplot as plt
plt.scatter(pc_coords[0], pc_coords[1], s=1, color='grey', alpha=0.3, label='Point cloud')
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
plt.plot(ra_model_plot, dec_model_plot, 'b-', linewidth=2, label='Model Streamline')

# matched model/data points in retained overlap
plt.scatter(
    ra_model_interp[valid], dec_model_interp[valid],
    s=25, label='Model at retained data arc lengths', color='blue', zorder=5
)
plt.scatter(
    ra_data[valid], dec_data[valid],
    s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
    label='Retained data points', zorder=6
)

plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
partitions = get_radial_bin_partitions(pc_coords, n_points)
ax = plt.gca()
plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
plt.gca().invert_xaxis()
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
plt.legend()

## Full fit streamline starts here

In [ ]:
def get_omega(mass, r0):
    '''
    this gets value of omega when r_cent = 0.5 * r0
    '''
    omega_squared = 0.5 * G * mass / (jnp.power(r0, 3) * jnp.power(au_in_km, 2)) # in s^-2
    omega = jnp.power(omega_squared, 0.5) # in s^-1
    return omega

opt_params = initial_opt_params.copy()

# Define physically reasonable bounds (omega bounds transformed to natural log space)
# These bounds are also used as normalization anchors: x_norm = (x - min) / (max - min).
# Provide bounds for every optimized parameter.
r0_min, r0_max = 200.0, 20000.0 # param bounds in au

# the omega bounds are set by keeping centrifugal radius reasonable (r_cent = 0.5 r0)
omega_max = get_omega(fixed_params['mass'], r0_min)
omega_min = get_omega(fixed_params['mass'], r0_max)
# print these in scientific notation for sanity check
print(f"Omega bounds: {omega_min:.2e} to {omega_max:.2e} 1/s")

param_bounds = {
    'r0': (r0_min, r0_max),                    # radius between 200-20000 au
    'theta0': (0.0, np.pi),                    # polar angle 0-pi
    'phi0': (0.0, 2*np.pi),                    # azimuthal angle 0-2pi
    'log_omega': (np.log(omega_min), np.log(omega_max)),  # omega in [omega_min, omega_max] 1/s
    'v_r0': (-5.0, 5.0),                       # radial velocity -5 to 5 km/s
}

log_file = 'streamfit_test_output/optimisation_log.csv'
trace_file = 'streamfit_test_output/optimisation_trace.csv'
trace_every = 1
n_epochs = 100
info_every = 10
learning_rate = 0.01 # Single learning rate applied to all normalized optimization parameters
loss_method = 'radecvel'

gradient_tol = 1e-2 * len(initial_opt_params) # gradient tolerance scaled by number of parameters

best_opt_params, loss_history, param_errors = gradient_descent.fit_streamline(
    opt_params,
    fixed_params,
    data,
    uncertainties,
    distance,
    learning_rate=learning_rate,
    param_bounds=param_bounds,
    n_epochs=n_epochs,
    info_every=info_every,
    loss_threshold=0.05,
    loss_threshold_epochs=5,
    gradient_tol=gradient_tol,
    gradient_tol_epochs=5,
    early_stopping_patience=40,
    log_file=log_file,
    trace_file=trace_file,
    trace_every=trace_every,
    loss_method=loss_method,
    output_uncertainties=True,
 )


print(f"Optimized using loss_method='{loss_method}'")

# Plot loss history
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Optimization Progress')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Final model with best-fit parameters
ra_best, dec_best, v_best = gradient_descent.forward_model(best_opt_params, fixed_params, distance)

# remove NaN values (due to rmin) from model for plotting
not_nan = ~jnp.isnan(ra_best) & ~jnp.isnan(dec_best) & ~jnp.isnan(v_best)
ra_best = ra_best[not_nan]
dec_best = dec_best[not_nan]
v_best = v_best[not_nan]

# plot it on top of the data
plt.scatter(pc_coords[0], pc_coords[1], s=1, color='gray', alpha=0.3, label='Point cloud', zorder=4)
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
plt.plot(ra_best, dec_best, color='blue', linewidth=2, label='Best-fit Model Streamline')

# get the model positions at data arc length points (overlap restricted)
ra_best_interp, dec_best_interp, v_best_interp, valid = gradient_descent.match_model_to_data_curve(
    ra_best, dec_best, v_best, ra_data, dec_data)

# plot model positions only where overlap is retained
plt.scatter(
    ra_best_interp[valid], dec_best_interp[valid],
    s=25, label='Model at retained data arc lengths', color='blue', zorder=5
)
plt.scatter(
    ra_data[valid], dec_data[valid],
    s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
    label='Retained data points', zorder=6
)

plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')

# Save axis limits before adding background/circles
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Fill background grey, then cut out the partition circles with white fill
import matplotlib.patches as patches
ax.set_facecolor('lightgrey')

partitions = get_radial_bin_partitions(pc_coords, n_points)

# Draw filled white circles for each partition to reveal the data area inside
for partition_radius in partitions:
    circle = patches.Circle((0, 0), partition_radius, 
                           facecolor='white', 
                           edgecolor='none', 
                           zorder=1)
    ax.add_patch(circle)

plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)

# Restore original axis limits
ax.set_xlim(xlim)
ax.set_ylim(ylim)

plt.gca().invert_xaxis()
plt.legend()
plt.title('Best-fit Streamline Model')
plt.show()

In [ ]:
# todo: here plot model and data and by-eye for comparison